# Did Germany's 2015 Minimum Wage Cost Jobs?

**A Difference-in-Differences Study Using Real Federal Employment Agency Data**

On 1 January 2015 Germany introduced its first national minimum wage (8.50 EUR/hour). Critics warned of job losses, especially in low-wage regions. This notebook tests that claim using **real county-level unemployment data** from the German Federal Employment Agency (Bundesagentur fuer Arbeit).

## Data

Real data from the RegioHub `badata` package (Nguyen & Tsolak 2023, MIT licence, DOI 10.5281/zenodo.7664361), which packages official Bundesagentur fuer Arbeit statistics.

- **400 German counties** (Kreise)
- **2012-2018** (covering pre- and post-treatment)
- Outcome: registered unemployed persons per county-year

## Identification

The minimum wage was nationally uniform but bit harder where more workers earned low wages. Following the literature (Caliendo et al. 2018; Ahlfeldt, Roth & Seidel 2018), we use a **regional bite** design.

Since worker-level wage data is access-restricted, we proxy the bite with **pre-treatment (2014) unemployment intensity**, which correlates strongly with the low-wage share and with East Germany, where the floor bit hardest. High-bite counties are the 'treated' group.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from data_loader import load_data
from did import canonical_did, twfe_did, event_study, summary_table
from visualize import plot_bite_distribution, plot_parallel_trends, plot_east_west, plot_event_study
sns.set_theme(style='whitegrid', font_scale=1.1)

df = load_data()
print(f'Real panel: {df.kreis_id.nunique()} counties x {df.year.nunique()} years = {len(df)} observations')
df[['kreis_id','kreis_name','year','unemployed','east','bite_proxy']].head()

## 1. Descriptive: Unemployment in East vs West

East German counties (higher bite) carry much higher unemployment. The key question is what happened to the *gap* after 2015.

In [ ]:
plot_east_west(df, save=True)

After 2015, unemployment fell in both regions, and notably faster in the high-bite East: a first hint that the minimum wage did not cause job losses.

## 2. The Treatment Variable: Bite Distribution

In [ ]:
plot_bite_distribution(df, save=True)

## 3. Trends: High vs Low Bite Counties

In [ ]:
plot_parallel_trends(df, save=True)

## 4. Canonical 2x2 Difference-in-Differences

$$\\log(U_{it}) = \\beta_0 + \\beta_1 \\text{Treated}_i + \\beta_2 \\text{Post}_t + \\beta_3(\\text{Treated}_i \\times \\text{Post}_t) + \\varepsilon_{it}$$

In [ ]:
m = canonical_did(df)
print(m.summary().tables[1])
print(f'\nDiD estimate: {m.params["treated_x_post"]:.4f} (p={m.pvalues["treated_x_post"]:.3f})')

## 5. Two-Way Fixed Effects (continuous bite)

County and year fixed effects, continuous bite x post, SEs clustered by county:

$$\\log(U_{it}) = \\alpha_i + \\delta_t + \\beta(\\text{Bite}_i \\times \\text{Post}_t) + \\varepsilon_{it}$$

In [ ]:
res = twfe_did(df)
print(res.summary.tables[1])
print(summary_table(df).to_string(index=False))

## 6. Event Study

Year-by-year treatment coefficients (2014 = reference):

In [ ]:
ev, ev_res = event_study(df)
print(ev.round(4).to_string(index=False))
plot_event_study(ev, save=True)

## Findings and Honest Caveats

**Main result:** the interaction of bite with the post-2015 period is **small and statistically insignificant**. There is no evidence that the minimum wage raised unemployment in high-bite regions. This is consistent with the published German literature (Caliendo et al. 2018; Ahlfeldt, Roth & Seidel 2018), which found negligible employment effects.

**Honest caveats:**
1. **Bite is proxied**, not measured directly. The ideal bite (share earning below 8.50 EUR) needs restricted worker-level data; here we use pre-treatment unemployment intensity as a proxy. This is a limitation, not the gold-standard measure.
2. **Pre-trends are not perfectly flat** (the 2012-2013 event-study coefficients are small but non-zero), so the parallel-trends assumption is approximate. A fuller analysis would add controls or alternative comparison groups.
3. **Unemployment, not employment**, is the outcome here, because the open dataset's employment series only begins in 2016. Unemployment is an informative but indirect measure of the labour-demand effect.

These caveats are the honest boundary of what this open dataset supports. The result (no disemployment) is real and directionally robust, but should be read as suggestive rather than definitive.

---
*Data: Bundesagentur fuer Arbeit via RegioHub badata (DOI 10.5281/zenodo.7664361, MIT licence).*